##Trabajo Práctico - Pipeline con Aerolinea
### Arquitectura Bronze, Silver y Gold con Pyspark y Delta Lake

En este notebook se construye un pipeline de datos para una aerolinea utilizando arquitectura por capas:
- *Bronze*: Ingesta de archivos fuente
- *Silver*: Limpieza, tipificación y deduplicación
- *Gold*: Generación de KPIs de negocio

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

* Catalogo Bronze


In [0]:
CATALOG = "airline_mantenimiento"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
VOLUME_NAME = "landing"


VUELOS_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/vuelos_diarios.csv"
AERONAVES_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/aeronaves.csv"
AEROPUERTOS_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/aeropuertos.csv"
MANTENIMIENTOS_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/mantenimientos_rds.csv"

spark = SparkSession.builder.getOrCreate()

In [0]:
#Creacion de  schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

#Creamos el volume
spark.sql(f"""CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}""")



DataFrame[]

#### 3. Capa Bronze 

En esta capa se leen los archivos CSV originales, se normalizan tipos basicos y se alamcena como tablas Delta

In [0]:
vuelos_raw = (spark.read.option("header",True).option("InferSchema",True).csv(VUELOS_PATH))
aeronaves_raw = (spark.read.option("header",True).option("InferSchema",True).csv(AERONAVES_PATH))
aeropuertos_raw = (spark.read.option("header",True).option("InferSchema",True).csv(AEROPUERTOS_PATH))
mantenimientos_raw = (spark.read.option("header",True).option("InferSchema",True).csv(MANTENIMIENTOS_PATH))

In [0]:
#Validacion 
print("filas vuelos: ", vuelos_raw.count())
print("filas aeronaves: ", aeronaves_raw.count())
print("filas aeropuertos: ", aeropuertos_raw.count())
print("filas mantenimientos: ", mantenimientos_raw.count())

filas vuelos:  100
filas aeronaves:  10
filas aeropuertos:  5
filas mantenimientos:  100


In [0]:
#Normalizar vuelos Bronze
vuelos_bronze = (vuelos_raw
                 .withColumn("fecha",F.to_date(F.col("fecha")))
                 .withColumn("vuelo_id", F.col("vuelo_id").cast("string"))
                 .withColumn("origen_id", F.col("origen_id").cast("string"))
                 .withColumn("destino_id", F.col("destino_id").cast("string"))
                 .withColumn("aeronave_id", F.col("aeronave_id").cast("string"))
                 .withColumn("estado", F.col("estado").cast("string"))
                 .withColumn("duracion_min", F.col("duracion_min").cast("int"))
                 .withColumn("ingestion_time", F.current_timestamp())
                )


In [0]:
aeronaves_bronze = (
    aeronaves_raw.withColumn("aeronave_id", F.col("aeronave_id").cast("string"))
    .withColumn("anio_fabricacion", F.col("anio_fabricacion").cast("int"))
    .withColumn("modelo", F.col("modelo").cast("string"))
    .withColumn("fabricante", F.col("fabricante").cast("string"))
)

In [0]:
aeropuertos_bronze = (aeropuertos_raw.withColumn("aeropuerto_id", F.col("aeropuerto_id").cast("string"))
                      .withColumn("lat", F.col("lat").cast("double"))
                      .withColumn("lon", F.col("lon").cast("double"))
                      .withColumn("nombre", F.col("nombre").cast("string"))
                      .withColumn("ciudad", F.col("ciudad").cast("string"))
                      .withColumn("pais", F.col("pais").cast("string"))
                   )


In [0]:
mantenimientos_bronze = (mantenimientos_raw.withColumn("mantenimiento_id", F.col("mantenimiento_id").cast("string"))
                         .withColumn("aeronave_id", F.col("aeronave_id").cast("string"))
                         .withColumn("fecha", F.to_date(F.col("fecha")))
                         .withColumn("tipo", F.col("tipo").cast("string"))
                         .withColumn("costo_usd", F.col("costo_usd").cast("double"))
                         .withColumn("duracion_hr", F.col("duracion_hr").cast("double"))
                   )

In [0]:
# Esta celda añade columnas de auditoría a las tablas Bronze:
# - ingestion_time: fecha y hora de carga
# - source_file: archivo origen de los datos
# Esto permite rastrear de dónde vienen los datos y cuándo fueron procesados.

vuelos_bronze = vuelos_bronze \
    .withColumn("ingestion_time", F.current_timestamp()) \
    .withColumn("source_file", F.input_file_name())

aeronaves_bronze = aeronaves_bronze \
    .withColumn("ingestion_time", F.current_timestamp()) \
    .withColumn("source_file", F.input_file_name())

aeropuertos_bronze = aeropuertos_bronze \
    .withColumn("ingestion_time", F.current_timestamp()) \
    .withColumn("source_file", F.input_file_name())

mantenimientos_bronze = mantenimientos_bronze \
    .withColumn("ingestion_time", F.current_timestamp()) \
    .withColumn("source_file", F.input_file_name())

In [0]:
#Guardar en tablas delta

vuelos_bronze.write.format("delta")\
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.vuelos_bronze")

aeronaves_bronze.write.format("delta")\
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.aeronaves_bronze")

aeropuertos_bronze.write.format("delta")\
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.aeropuertos_bronze")

mantenimientos_bronze.write.format("delta")\
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.mantenimientos_bronze")


In [0]:
spark.sql(f"select * from {CATALOG}.{BRONZE_SCHEMA}.vuelos_bronze").display()

vuelo_id,fecha,origen_id,destino_id,aeronave_id,estado,duracion_min,ingestion_time
V0001,2024-05-21,MAD,JFK,A005,a_tiempo,288,2026-04-10T20:15:39.718Z
V0002,2024-05-05,MAD,LHR,A002,cancelado,492,2026-04-10T20:15:39.718Z
V0003,2024-05-02,MAD,JFK,A004,a_tiempo,577,2026-04-10T20:15:39.718Z
V0004,2024-05-20,MAD,LHR,A004,cancelado,618,2026-04-10T20:15:39.718Z
V0005,2024-05-14,BCN,BCN,A010,retrasado,66,2026-04-10T20:15:39.718Z
V0006,2024-05-25,BCN,BCN,A006,retrasado,219,2026-04-10T20:15:39.718Z
V0007,2024-05-07,JFK,JFK,A002,retrasado,159,2026-04-10T20:15:39.718Z
V0008,2024-05-12,JFK,LHR,A005,a_tiempo,530,2026-04-10T20:15:39.718Z
V0009,2024-05-18,MAD,BCN,A002,cancelado,360,2026-04-10T20:15:39.718Z
V0010,2024-05-27,CDG,CDG,A010,a_tiempo,131,2026-04-10T20:15:39.718Z



#   4. Capa Silver 
En esta capa se limpian los datos, se tipifican 


In [0]:
# =========================================================
# Normalización previa para Silver
# =========================================================
# Esta celda limpia campos clave antes de construir Silver:
# - elimina espacios
# - estandariza texto
# - normaliza el estado de vuelos
# Esto reduce errores en joins, filtros y agregaciones.

vuelos_bz = vuelos_bz \
    .withColumn("origen_id", F.trim(F.col("origen_id"))) \
    .withColumn("destino_id", F.trim(F.col("destino_id"))) \
    .withColumn("aeronave_id", F.trim(F.col("aeronave_id"))) \
    .withColumn("estado", F.lower(F.trim(F.col("estado"))))

aeronaves_bz = aeronaves_bz \
    .withColumn("aeronave_id", F.trim(F.col("aeronave_id"))) \
    .withColumn("modelo", F.trim(F.col("modelo"))) \
    .withColumn("fabricante", F.trim(F.col("fabricante")))

aeropuertos_bz = aeropuertos_bz \
    .withColumn("aeropuerto_id", F.trim(F.col("aeropuerto_id"))) \
    .withColumn("nombre", F.trim(F.col("nombre"))) \
    .withColumn("ciudad", F.trim(F.col("ciudad"))) \
    .withColumn("pais", F.trim(F.col("pais")))

mantenimientos_bz = mantenimientos_bz \
    .withColumn("aeronave_id", F.trim(F.col("aeronave_id"))) \
    .withColumn("tipo", F.trim(F.col("tipo")))

In [0]:
vuelos_bz = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.vuelos_bronze")
aeronaves_bz = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.aeronaves_bronze")
aeropuertos_bz = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.aeropuertos_bronze")
mantenimientos_bz = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.mantenimientos_bronze")

In [0]:
#Transformaciones Silver

vuelos_silver = (
    vuelos_bz
    .select("vuelo_id", "fecha", "origen_id", "destino_id", "aeronave_id", "estado", "duracion_min")
    .withColumn("fecha", F.to_date(F.col("fecha")))
    .withColumn("duracion_min", F.col("duracion_min").cast("int"))
    .dropDuplicates()
)

aeronaves_silver = (
    aeronaves_bz
    .select("aeronave_id", "modelo", "fabricante", "anio_fabricacion")
    .dropDuplicates()
)

aeropuertos_silver = (
    aeropuertos_bz
    .select("aeropuerto_id", "nombre", "ciudad", "pais", "lat", "lon")
    .dropDuplicates()
)

mantenimientos_silver = (
    mantenimientos_bz
    .select("mantenimiento_id", "aeronave_id", "fecha", "tipo", "costo_usd", "duracion_hr")
    .withColumn("fecha", F.to_date(F.col("fecha")))
    .withColumn("costo_usd", F.col("costo_usd").cast("double"))
    .withColumn("duracion_hr", F.col("duracion_hr").cast("double"))
    .dropDuplicates()
)


In [0]:
# =========================================================
# Validaciones de calidad Silver
# =========================================================
# Esta celda revisa calidad de datos en Silver:
# - nulos en campos clave
# - duplicados
# - valores negativos
# - estados inválidos
# Sirve para medir si la limpieza fue suficiente antes de publicar la capa Silver.

estados_validos = ["a_tiempo", "retrasado", "cancelado"]

print("========== VALIDACIONES VUELOS ==========")
print("nulos vuelo_id:", vuelos_silver.filter(F.col("vuelo_id").isNull()).count())
print("nulos origen_id:", vuelos_silver.filter(F.col("origen_id").isNull()).count())
print("nulos destino_id:", vuelos_silver.filter(F.col("destino_id").isNull()).count())
print("nulos aeronave_id:", vuelos_silver.filter(F.col("aeronave_id").isNull()).count())
print("fechas nulas:", vuelos_silver.filter(F.col("fecha").isNull()).count())
print("duracion negativa:", vuelos_silver.filter(F.col("duracion_min") < 0).count())
print("duplicados vuelo_id:", vuelos_silver.groupBy("vuelo_id").count().filter(F.col("count") > 1).count())
print("estados inválidos:", vuelos_silver.filter(~F.col("estado").isin(estados_validos)).count())

print("========== VALIDACIONES AERONAVES ==========")
print("nulos aeronave_id:", aeronaves_silver.filter(F.col("aeronave_id").isNull()).count())
print("anios nulos:", aeronaves_silver.filter(F.col("anio_fabricacion").isNull()).count())

print("========== VALIDACIONES AEROPUERTOS ==========")
print("nulos aeropuerto_id:", aeropuertos_silver.filter(F.col("aeropuerto_id").isNull()).count())
print("nulos nombre:", aeropuertos_silver.filter(F.col("nombre").isNull()).count())

print("========== VALIDACIONES MANTENIMIENTOS ==========")
print("nulos mantenimiento_id:", mantenimientos_silver.filter(F.col("mantenimiento_id").isNull()).count())
print("nulos aeronave_id:", mantenimientos_silver.filter(F.col("aeronave_id").isNull()).count())
print("fechas nulas:", mantenimientos_silver.filter(F.col("fecha").isNull()).count())
print("costo negativo:", mantenimientos_silver.filter(F.col("costo_usd") < 0).count())
print("duracion negativa:", mantenimientos_silver.filter(F.col("duracion_hr") < 0).count())

========== VALIDACIONES VUELOS ==========
nulos vuelo_id: 0
nulos origen_id: 0
nulos destino_id: 0
nulos aeronave_id: 0
fechas nulas: 0
duracion negativa: 0
duplicados vuelo_id: 0
estados inválidos: 0
========== VALIDACIONES AERONAVES ==========
nulos aeronave_id: 0
anios nulos: 0
========== VALIDACIONES AEROPUERTOS ==========
nulos aeropuerto_id: 0
nulos nombre: 0
========== VALIDACIONES MANTENIMIENTOS ==========
nulos mantenimiento_id: 0
nulos aeronave_id: 0
fechas nulas: 0
costo negativo: 0
duracion negativa: 0


In [0]:
# =========================================================
# Registros rechazados de vuelos
# =========================================================
# Esta celda separa los registros inválidos de vuelos para auditoría:
# - vuelo_id nulo
# - fecha nula
# - duración negativa
# - estado inválido
# Así se pueden revisar errores sin perder trazabilidad.

estados_validos = ["a_tiempo", "retrasado", "cancelado"]

vuelos_rechazados = vuelos_silver.filter(
    F.col("vuelo_id").isNull() |
    F.col("fecha").isNull() |
    (F.col("duracion_min") < 0) |
    (~F.col("estado").isin(estados_validos))
)

print("total registros rechazados de vuelos:", vuelos_rechazados.count())
vuelos_rechazados.display()

total registros rechazados de vuelos: 0


vuelo_id,fecha,origen_id,destino_id,aeronave_id,estado,duracion_min


In [0]:
#Validacion de calidad

print("nulos vuelo_id", vuelos_silver.filter(F.col("vuelo_id").isNull()).count())

print("Duplicados vuelos", vuelos_silver.count() - vuelos_silver.dropDuplicates().count())


nulos vuelo_id 0
Duplicados vuelos 0


In [0]:
#Guardar en tablas delta

#Guardar en tablas delta Silver

vuelos_silver.write.format("delta")\
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.vuelos")

aeronaves_silver.write.format("delta")\
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.aeronaves")

aeropuertos_silver.write.format("delta")\
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.aeropuertos")

mantenimientos_silver.write.format("delta")\
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.mantenimientos")

In [0]:
# =========================================================
# Guardar tabla de rechazados
# =========================================================
# Esta celda guarda los registros inválidos detectados en Silver
# para análisis posterior de calidad de datos.

vuelos_rechazados.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.vuelos_rechazados")

In [0]:
print("nulos vuelo_id", vuelos_silver.filter(F.col("vuelo_id").isNull()).count())



nulos vuelo_id 0


## 5. Capa Gold - Indicadores de negocio

En esta capa se construyen indicadores analíticos a partir de los datos depurados de Silver.  
El objetivo es generar métricas de negocio que permitan analizar la puntualidad de los vuelos, los costos de mantenimiento y las rutas más frecuentes.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Asegurar schema Gold
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

# Leer tablas desde Silver
df_vuelos = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.vuelos")
df_aeropuertos = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.aeropuertos")
df_aeronaves = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.aeronaves")
df_mantenimientos = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.mantenimientos")


### 5.1 Preparación de dimensiones

En esta etapa se preparan las dimensiones necesarias para enriquecer la información de vuelos y mantenimientos, incorporando datos de aeropuertos y aeronaves.

In [0]:
# Aeropuerto de origen
df_aeropuertos_origen = df_aeropuertos.select(
    F.col("aeropuerto_id").alias("origen_id"),
    F.col("nombre").alias("origen_nombre"),
    F.col("ciudad").alias("origen_ciudad"),
    F.col("pais").alias("origen_pais")
)

# Aeropuerto de destino
df_aeropuertos_destino = df_aeropuertos.select(
    F.col("aeropuerto_id").alias("destino_id"),
    F.col("nombre").alias("destino_nombre"),
    F.col("ciudad").alias("destino_ciudad"),
    F.col("pais").alias("destino_pais")
)

In [0]:
df_vuelos_enriquecido = (
    df_vuelos
    .join(df_aeropuertos_origen, on="origen_id", how="left")
    .join(df_aeropuertos_destino, on="destino_id", how="left")
    .join(df_aeronaves, on="aeronave_id", how="left")
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
)

### 5.2 KPI de puntualidad de vuelos

Se construye un indicador que resume la operación de vuelos por modelo, fabricante, país de origen y periodo, permitiendo medir la puntualidad y el comportamiento operativo.

In [0]:
df_kpi_puntualidad = (
    df_vuelos_enriquecido
    .groupBy("modelo", "fabricante", "origen_pais", "mes", "anio")
    .agg(
        F.count("vuelo_id").alias("total_vuelos"),
        F.count(F.when(F.col("estado") == "a_tiempo", True)).alias("vuelos_a_tiempo"),
        F.count(F.when(F.col("estado") == "retrasado", True)).alias("vuelos_retrasados"),
        F.count(F.when(F.col("estado") == "cancelado", True)).alias("vuelos_cancelados"),
        F.round(F.avg("duracion_min"), 2).alias("duracion_promedio_min")
    )
    .withColumn(
        "puntualidad_pct",
        F.round((F.col("vuelos_a_tiempo") / F.col("total_vuelos")) * 100, 2)
    )
)

In [0]:
df_kpi_puntualidad.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.kpi_puntualidad_vuelos")

### 5.3 KPI de costos de mantenimiento

Se construye un indicador analítico sobre los mantenimientos realizados a las aeronaves, incluyendo frecuencia, costo y duración por tipo de mantenimiento y periodo.

In [0]:
df_mantenimientos_enriquecido = (
    df_mantenimientos
    .join(df_aeronaves, on="aeronave_id", how="left")
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
)

In [0]:
df_kpi_mantenimiento = (
    df_mantenimientos_enriquecido
    .groupBy("aeronave_id", "modelo", "fabricante", "tipo", "mes", "anio")
    .agg(
        F.count("mantenimiento_id").alias("total_mantenimientos"),
        F.round(F.avg("costo_usd"), 2).alias("costo_promedio_usd"),
        F.round(F.sum("costo_usd"), 2).alias("costo_total_usd"),
        F.round(F.avg("duracion_hr"), 2).alias("duracion_promedio_hr"),
        F.round(F.sum("duracion_hr"), 2).alias("duracion_total_hr")
    )
)

In [0]:
df_kpi_mantenimiento.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.kpi_costo_mantenimiento")

### 5.4 KPI de rutas más frecuentes

Se construye un indicador que identifica las rutas con mayor cantidad de vuelos, relacionando aeropuertos de origen y destino.

In [0]:
df_kpi_rutas_frecuentes = (
    df_vuelos_enriquecido
    .groupBy(
        "origen_nombre",
        "origen_ciudad",
        "origen_pais",
        "destino_nombre",
        "destino_ciudad",
        "destino_pais"
    )
    .agg(
        F.count("vuelo_id").alias("total_vuelos")
    )
    .orderBy(F.desc("total_vuelos"))
)

In [0]:
df_kpi_rutas_frecuentes.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.kpi_rutas_frecuentes")

### 5.5 Validación final de la capa Gold

Finalmente, se consultan las tablas generadas en Gold para validar que los indicadores fueron construidos y almacenados correctamente.

In [0]:
spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}").display()

spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.kpi_puntualidad_vuelos").display()
spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.kpi_costo_mantenimiento").display()
spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.kpi_rutas_frecuentes").display()

database,tableName,isTemporary
gold,kpi_costo_mantenimiento,false
gold,kpi_puntualidad_vuelos,false
gold,kpi_rutas_frecuentes,false


modelo,fabricante,origen_pais,mes,anio,total_vuelos,vuelos_a_tiempo,vuelos_retrasados,vuelos_cancelados,duracion_promedio_min,puntualidad_pct
A350,Airbus,Francia,5,2024,6,2,2,2,490.67,33.33
A320,Airbus,Reino Unido,5,2024,4,3,0,1,338.0,75.0
B777,Boeing,Reino Unido,5,2024,4,4,0,0,266.5,100.0
A350,Airbus,España,5,2024,12,3,6,3,350.5,25.0
E190,Embraer,Francia,5,2024,2,0,1,1,497.0,0.0
E190,Embraer,Reino Unido,5,2024,1,0,1,0,600.0,0.0
B777,Boeing,Francia,5,2024,7,3,1,3,395.43,42.86
B737,Boeing,Francia,5,2024,6,2,0,4,293.33,33.33
A350,Airbus,Reino Unido,5,2024,4,1,2,1,353.5,25.0
B737,Boeing,Reino Unido,5,2024,5,1,3,1,299.2,20.0


aeronave_id,modelo,fabricante,tipo,mes,anio,total_mantenimientos,costo_promedio_usd,costo_total_usd,duracion_promedio_hr,duracion_total_hr
A007,A350,Airbus,preventivo,5,2024,2,14911.75,29823.5,6.15,12.3
A009,A320,Airbus,preventivo,5,2024,4,15293.45,61173.78,15.33,61.3
A010,B737,Boeing,correctivo,5,2024,3,36064.58,108193.73,9.5,28.5
A002,B737,Boeing,inspección,5,2024,1,44255.92,44255.92,13.8,13.8
A007,A350,Airbus,inspección,5,2024,6,40718.73,244312.38,12.02,72.1
A010,B737,Boeing,inspección,5,2024,2,37805.7,75611.39,3.1,6.2
A001,B777,Boeing,correctivo,5,2024,5,28187.36,140936.81,17.74,88.7
A009,A320,Airbus,inspección,5,2024,3,13276.73,39830.19,13.2,39.6
A004,B737,Boeing,inspección,5,2024,1,20530.13,20530.13,5.8,5.8
A006,A350,Airbus,correctivo,5,2024,2,37393.67,74787.34,14.05,28.1


origen_nombre,origen_ciudad,origen_pais,destino_nombre,destino_ciudad,destino_pais,total_vuelos
Josep Tarradellas Barcelona-El Prat,Barcelona,España,Josep Tarradellas Barcelona-El Prat,Barcelona,España,7
Adolfo Suárez Madrid-Barajas,Madrid,España,Charles de Gaulle,París,Francia,7
Charles de Gaulle,París,Francia,Charles de Gaulle,París,Francia,6
Adolfo Suárez Madrid-Barajas,Madrid,España,John F. Kennedy International,Nueva York,EEUU,6
Heathrow,Londres,Reino Unido,John F. Kennedy International,Nueva York,EEUU,6
Charles de Gaulle,París,Francia,John F. Kennedy International,Nueva York,EEUU,6
Josep Tarradellas Barcelona-El Prat,Barcelona,España,Charles de Gaulle,París,Francia,5
Charles de Gaulle,París,Francia,Josep Tarradellas Barcelona-El Prat,Barcelona,España,5
Adolfo Suárez Madrid-Barajas,Madrid,España,Adolfo Suárez Madrid-Barajas,Madrid,España,4
Adolfo Suárez Madrid-Barajas,Madrid,España,Josep Tarradellas Barcelona-El Prat,Barcelona,España,4
